# One reader card, or a writer that can look back?

**Guiding question:** When the answer is a different sequence, does the writer need every reader
note, or is one learned summary enough?

You can carry an ordered stream. Now the whole input is available before writing begins.
Let one component read it and another write the answer. The reader is the **encoder**.
The writer is the **decoder**. These are roles, not names for a particular internal layer.
Predicting a separate ordered answer is a **sequence-to-sequence** task.
The [existing 02C lesson](../02-tokenization-and-embeddings/02C-context-windows-to-static-embeddings.ipynb)
introduced these roles. Here you test the interface on a held-out split.

Why leave color sentences? Continuing that stream does not require a separately aligned answer.
Reverse `[3, 1, 4, 1]` into `[1, 4, 1, 3]` instead. Every needed source address is known.
This is [Stage 04's routing microscope](../../04-transformers/05-encoder-decoder-and-cross-attention-theory.md).
It avoids a second text tokenizer while the task changes.

| Attempt | Complaint | Evidence |
|---|---|---|
| Correct previous digits in practice | Generation lacks those answers | Alignment table and wrong-guess trace |
| One recurrent summary | Writer cannot reopen notes | Accuracy by source length |
| Writer asks every source note | Reading is still sequential | Same split, measured map, generation |

Prerequisites: tensor training, running notes, and token-row lookups. All state is rebuilt here.
No downloads or checkpoints. Small CPU tensors, one thread, two bounded training runs.
[Handwritten companion](03-recurrent-encoder-decoder-to-attention-theory.md).

## 1. Read one sequence, write another

**Predict:** Can an order-sensitive reader still have a narrow handoff?

![The reader compresses an ordered source into one learned card. The writer cannot reopen earlier source positions.](images/03-evo-04-one-card-bottleneck.png)
*An ordered reader may give the writer only one source card.*


```text
reader: 3 -> note -> 1 -> note -> 4 -> note -> 1 -> final card
writer: final card + start -> 1 -> 4 -> 1 -> 3 -> end
```

A learned recurrent summary is not an average of token rows. The reader updates its note in order.
Only a fixed number of note entries crosses the interface, regardless of source length.
Short sources may fit well. A fixed-summary model is not guaranteed to fail.

## 2. Practice and generation give different help

**Predict:** Should the writer receive the current answer or the previous correct answer?
During practice, supply the correct **previous** digit. This is **teacher forcing**.
`start` is a beginning marker; `end` is a stopping answer. The current answer never appears as
its own input.

| Write step | Practice input | Expected answer |
|---|---|---|
| First | start | 1 |
| Second | 1 | 4 |
| Third | 4 | 1 |
| Fourth | 1 | 3 |
| Fifth | 3 | end |

Generation cannot borrow correct previous answers. It feeds its own guesses back.
If the first guess is 4 instead of 1, the second input becomes 4 instead of 1.
That changes the next computation. It does not prove every error must cascade.

![Practice supplies the correct previous answer. Generation feeds back its own guess, changing the next input.](images/03-evo-06-practice-vs-generation.png)
*Practice and generation use different previous answers.*


In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)
started = time.perf_counter()

**Your turn:** Change only `first_guess`. Trace the next input before the training loop exists.

In [ ]:
first_guess = 4  # CHANGE THIS: 1 is the correct first answer
correct_target = [1, 4, 1, 3, "end"]
practice_inputs = ["start"] + correct_target[:-1]
print("Practice input -> answer:", list(zip(practice_inputs, correct_target)))
print("Generation: start -> guessed", first_guess, "-> next input", first_guess)
print("Correct previous input at step two:", practice_inputs[1])
print("Takeaway: a wrong guess changes the next input, not the stored source.")

**Reflection / complaint:** Practice sees correct history. Generation sees model history.
This mismatch is called **exposure bias**. Teacher forcing is a training method, not an
architecture requirement. It also applies to models that continue one stream.
Report both conditions rather than calling practice accuracy generation quality.

## 3. Give both models exactly the same examples

**Predict:** Will the one-card reader fail on every short source? Do not assume it.
Digits 1 through 8 are token IDs: addresses for trainable rows, not number magnitudes.
ID 0 is reserved for padding, 9 means start, and 10 means end.
An **embedding** is the trainable row retrieved by one address.
Each batch uses one source length, so it needs no padding. Both models train on lengths 2 through 8
and use the same held-out rows at each length. This tests unseen sequences at trained lengths,
not extrapolation to longer lengths. The small length-2 space limits distinct rows.

In [ ]:
PAD,BOS,EOS,V=0,9,10,11
rng=random.Random(31)
train,test={},{}
for length in range(2,9):
    unique=set()
    count=min(1200,8**length-40)
    while len(unique)<count+40:
        unique.add(tuple(rng.randint(1,8) for _ in range(length)))
    rows=list(sorted(unique)); rng.shuffle(rows)
    train[length]=torch.tensor(rows[:count]); test[length]=torch.tensor(rows[count:])

In [ ]:
for length in train:
    training_rows = {tuple(r) for r in train[length].tolist()}
    testing_rows = {tuple(r) for r in test[length].tolist()}
    assert training_rows.isdisjoint(testing_rows)
    print(f"Length {length}: {len(training_rows)} train, {len(testing_rows)} held out")
print("Takeaway: both models receive the same split; no test source appears in training.")

**Your turn:** Change only `example_length` to inspect another length bucket before fitting.

In [ ]:
example_length = 4  # CHANGE THIS: 2 through 8
example = test[example_length][0].tolist()
print("source:", example, "target:", list(reversed(example)) + [EOS])

**Reflection / complaint:** Examples are controlled. Capacities will still differ; report them.

## 4. Build a learned recurrent reader and writer

**Predict:** What changes if the writer keeps the final card but can reopen earlier notes?
A **gated recurrent unit (GRU)** is a running-note update with learned controls over retained and
new information. Gate details are not this interface test. The reader is a real GRU, not mean
pooling. Both models start their writer from its final note. The fixed-summary writer has no other
source route.

The second model lets the writer ask every source state for a useful detail. This is **attention**.
The writer state supplies a **query**, a learned question row. Each reader state is compared with
that question and remains available for retrieval. Higher match scores get larger positive weights.
`softmax` converts scores into weights that sum to one. A weighted blend becomes extra input and
extra output evidence for this writing step.

![A recurrent writer looks back at all source states. End-to-start is the expected reversal route, not a fabricated measured map.](images/03-evo-05-writer-looks-back.png)
*Each writing step can ask all reader notes for a useful detail.*


In [ ]:
class ReaderWriter(nn.Module):
    def __init__(self,look_back):
        super().__init__(); self.look_back=look_back
        self.embedding=nn.Embedding(V,24)
        self.reader=nn.GRU(24,64,batch_first=True)
        self.writer=nn.GRUCell(24+(64 if look_back else 0),64)
        self.query=nn.Linear(64,64,bias=False) if look_back else None
        self.head=nn.Linear(128 if look_back else 64,V)
    def read(self,source):
        memory,last=self.reader(self.embedding(source)); return memory,last[0]
    def step(self,previous,state,memory):
        if self.look_back:
            scores=torch.bmm(memory,self.query(state).unsqueeze(2)).squeeze(2)/8
            weights=scores.softmax(-1)
            context=torch.bmm(weights.unsqueeze(1),memory).squeeze(1)
            state=self.writer(torch.cat([self.embedding(previous),context],-1),state)
            logits=self.head(torch.cat([state,context],-1))
        else:
            state=self.writer(self.embedding(previous),state); logits=self.head(state); weights=None
        return logits,state,weights
    def forward(self,source,teacher=True):
        memory,state=self.read(source); previous=torch.full((len(source),),BOS)
        target=torch.cat([source.flip(1),torch.full((len(source),1),EOS)],1) if teacher else None
        scores=[]; maps=[]
        for t in range(source.shape[1] + 1):
            logits,state,weights=self.step(previous,state,memory); scores.append(logits)
            if weights is not None: maps.append(weights)
            previous=target[:,t] if teacher else logits.argmax(-1)
        return torch.stack(scores,1), torch.stack(maps,1) if maps else None

**Walkthrough:** `read` returns all source states and the final state. `step` reads the previous
target token and updates the writer note. With look-back enabled, it first matches the writer query
against every source state, blends those states, and supplies that blend to the writer and output
layer. Dividing by 8 keeps match scale moderate for width 64. The last method repeats this step.
Correct previous answers enter only in practice mode; generation uses only the last model choice.

No source state is averaged before the comparison. Attention creates a new weighted blend each
writing step. Reader and writer updates remain recurrent, hence sequential.
If you later pad mixed-length batches, hide blank source slots before calculating weights.
Separately, do not grade blank targets. Reading masks and loss masks solve different problems.
Equal-length batches here avoid both without silently ignoring them.

In [ ]:
models, histories = {}, {}
for look_back in [False, True]:
    torch.manual_seed(31)
    model = ReaderWriter(look_back)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
    schedule = random.Random(99)
    batch_rng = torch.Generator().manual_seed(99)
    history = []
    for update in range(1000):
        length = schedule.randint(2, 8)
        rows = train[length]
        source = rows[torch.randint(len(rows), (64,), generator=batch_rng)]
        target = torch.cat([source.flip(1), torch.full((64, 1), EOS)], 1)
        logits, _ = model(source, teacher=True)
        loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        history.append(loss.item())
    model.eval()
    models[look_back], histories[look_back] = model, history
    print("look back:", look_back, "loss", round(history[0], 4), "->", round(history[-1], 4),
          "parameters", sum(p.numel() for p in model.parameters()))

**Walkthrough:** Both runs restart the same batch schedule and optimizer settings, receiving the
same examples in the same order. Each practice update includes end. The loss teaches the reader
and writer. Parameter counts differ; one seed does not isolate all capacity or optimization effects.

**Your turn:** Change only `look_back_to_inspect`. Both printed models still have a recurrent reader.

In [ ]:
look_back_to_inspect = False  # CHANGE THIS: True
print(models[look_back_to_inspect])

**Reflection / complaint:** A falling practice loss is not a generated reversed sequence.

## 5. Grade complete answers, by source length

**Predict:** Which model will reverse more held-out rows? Will practice overstate generation?
**Sequence accuracy** requires every answer token, including end, to be correct.
One wrong token fails the whole row. Practice uses correct previous answers; free-running mode
uses its own choices. Generation begins at start and has no target inputs.
We allow source length plus one steps, a safety budget including end; source length is available
at inference. End must appear exactly after the reversed digits. Earlier end fails the row.
Continuing the budget after end cannot rescue a wrong row's exact score.

Both conditions can have equal exact accuracy: after the first wrong guess, the row already fails
either score. Equality would not mean their later histories are the same.

In [ ]:
results = []
with torch.no_grad():
    for look_back, model in models.items():
        for length, source in test.items():
            target = torch.cat([source.flip(1), torch.full((len(source), 1), EOS)], 1)
            tf_scores, _ = model(source, teacher=True)
            free_scores, _ = model(source, teacher=False)
            tf_ok = (tf_scores.argmax(-1) == target).all(1).float().mean().item()
            free_ok = (free_scores.argmax(-1) == target).all(1).float().mean().item()
            tf_token = (tf_scores.argmax(-1) == target).float().mean().item()
            free_token = (free_scores.argmax(-1) == target).float().mean().item()
            results.append((look_back, length, tf_ok, free_ok, tf_token, free_token))
print("look-back length practice-sequence free-sequence practice-token free-token")
for row in results:
    print(row[0], row[1], *[f"{value:.1%}" for value in row[2:]])
fixed_mean = np.mean([r[3] for r in results if not r[0]])
attention_mean = np.mean([r[3] for r in results if r[0]])
print(f"Mean over lengths, free-running: one card {fixed_mean:.1%}; look-back {attention_mean:.1%}")
if attention_mean > fixed_mean:
    print("Takeaway: look-back improved this held-out run; this is not a universal ranking.")
else:
    print("Takeaway: look-back did not improve this run; access exists but payoff was not established.")
fig, ax = plt.subplots(figsize=(6, 3))
for mode, label, color in [(False, "One learned card", "#ef8a23"), (True, "Writer looks back", "#2c7fb8")]:
    rows = [r for r in results if r[0] == mode]
    ax.plot([r[1] for r in rows], [r[3] for r in rows], "o-", label=label, color=color)
ax.set(xlabel="Source length", ylabel="Complete free-running accuracy", ylim=(0, 1.05))
ax.legend(); plt.show()

**Walkthrough:** Each held-out bucket is evaluated twice without weight updates. Exact matching
grades whole sequences. Token matching explains why equal exact scores can hide different later
behavior. The plot uses measured free-running values only.

**Your turn:** Change only the first digit in `source_to_reverse`. Generate without seeing the answer.
This probe is not automatically independent of training data.

In [ ]:
source_to_reverse = [3, 1, 4, 1]  # CHANGE THIS: first digit only, within 1..8
source = torch.tensor([source_to_reverse])
with torch.no_grad():
    for mode, model in models.items():
        scores, _ = model(source, teacher=False)
        print("look back", mode, "generated IDs:", scores.argmax(-1)[0].tolist())
print("Expected:", list(reversed(source_to_reverse)) + [EOS])

**Reflection / complaint:** A score says whether it worked, not how source positions were used.

## 6. Inspect the route without declaring victory from a picture

**Predict:** For `[3, 1, 4, 1]`, should reversal read source positions from last to first?
Two `1` tokens make content alone an ambiguous address. Recurrent reader states contain ordered
context. The learned map need not be a perfect anti-diagonal, a line running from upper right to
lower left. This heatmap comes from real free-running query weights, not hand-drawn target routing.

In [ ]:
with torch.no_grad():
    probe = torch.tensor([[3, 1, 4, 1]])
    logits, weights = models[True](probe, teacher=False)
    generated = logits.argmax(-1)[0].tolist()
    heat = weights[0].numpy()
fig, ax = plt.subplots(figsize=(6, 3))
image = ax.imshow(heat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(4), ["3 @ 0", "1 @ 1", "4 @ 2", "1 @ 3"])
ax.set_yticks(range(5), [f"step {i}: chose {digit}" for i, digit in enumerate(generated)])
ax.set(xlabel="Reader source position", ylabel="Writer step", title="Measured weights: diagnostic, not proof")
fig.colorbar(image, ax=ax, label="Read weight"); plt.show()
print("Row sums:", heat.sum(1).round(6).tolist())
print("Most-read source positions:", heat.argmax(1).tolist())
print("Expected digit addresses:", [3, 2, 1, 0], "; end has no single source address")
print("Takeaway: inspect routing beside generated answers; the picture alone cannot prove correctness.")

**Your turn:** Change only `writing_step`. Does its most-read source address match the reversal address?

In [ ]:
writing_step = 0  # CHANGE THIS: 1, 2, or 3
print("Most-read:", int(heat[writing_step].argmax()), "expected:", 3 - writing_step)

**Reflection / complaint:** A plausible route is useful evidence, not a complete causal explanation.

## 7. Evidence scorecard and the next complaint

| Question | Evidence |
|---|---|
| Was the one-card model recurrent? | Learned GRU final state alone initializes its writer |
| Was look-back real? | Writer queries all retained source states at each output step |
| Did it help? | Per-length and mean free-running scores with conditional takeaway |
| Is practice generation? | Both exact-sequence and token scores printed separately |
| Does the map prove the answer? | No; it diagnoses one route beside behavior |

| Teaching mechanism | Production counterpart | What changes | What stays |
|---|---|---|---|
| One reader card | Recurrent fixed-summary model | Larger state and data | No direct source addresses |
| Writer asks reader states | Recurrent attention | Scoring choices and masks | New source blend per step |
| Correct previous practice token | Teacher-forced training | Vocabulary and batch size | Generation uses own history |
| Recurrent reader/writer | Transformer encoder-decoder | Self-attention replaces recurrence | Separate reading/writing roles |

In [ ]:
print({"fixed_summary_mean_free_sequence_accuracy": round(float(fixed_mean), 6),
       "attention_mean_free_sequence_accuracy": round(float(attention_mean), 6),
       "held_out_sequences_per_length": len(test[2]), "source_lengths": list(test)})
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")

### Three-tier coverage ledger

- **Built and measured:** practice table, wrong-guess input trace, disjoint split, learned GRU
  fixed-summary reader/writer, query-based recurrent source attention, matched batch schedule,
  parameter counts, per-length practice/free-running sequence and token accuracy, generated probes,
  and measured source-routing heatmap.
- **Explained or illustrated:** teacher forcing and exposure bias, one-card limitation,
  source masks versus loss masks, gates versus source access, repeated-digit ambiguity,
  bidirectional encoders (a complete source may use both directions, unlike generated history),
  and attention predating Transformers.
- **Named and out of scope:** LSTM gate internals (existing memory lesson), beam search
  (keep several candidate answers), scheduled sampling (vary practice history), extrapolation,
  multi-seed rankings, production translation, and [Transformer internals](../../04-transformers/README.md).

**Failure checks:** do not substitute mean pooling for a learned recurrent summary;
do not feed the current answer as its own input; do not grade or read blank slots;
do not call practice generation; do not mistake a pretty map for proof.

Attention predates Transformers. LSTM improves fading, not the one-card interface.
A gated reader can still hand the writer only one card.

**Closing complaint:** Looking back changes source access, but recurrent reading remains sequential.
[Stage 04](../../04-transformers/README.md) asks: **what if we drop recurrence and let attention do all the reading?**